# Assistente de voz Ollama com Python e um toque de Javascript
##### Este notebook cria um fluxo de voz → texto → modelo (Ollama) → voz


In [ ]:
language ="pt"

# Gravação de Áudio

In [ ]:
from IPython.display import Audio, display, Javascript
from google.colab import output
from base64 import b64decode

# Código JavaScript para gravar áudio usando MediaStream Recording API
RECORD = """
const sleep  = time => new Promise(resolve => setTimeout(resolve, time))
const b2text = blob => new Promise(resolve => {
  const reader = new FileReader()
  reader.onloadend = e => resolve(e.srcElement.result)
  reader.readAsDataURL(blob)
})
var record = time => new Promise(async resolve => {
  stream = await navigator.mediaDevices.getUserMedia({ audio: true })
  recorder = new MediaRecorder(stream)
  chunks = []
  recorder.ondataavailable = e => chunks.push(e.data)
  recorder.start()
  await sleep(time)
  recorder.onstop = async ()=>{
    blob = new Blob(chunks)
    text = await b2text(blob)
    resolve(text)
  }
  recorder.stop()
})
"""

def record(sec=5):
    # Executa o código JavaScript para gravar o áudio
    display(Javascript(RECORD))
    js_result = output.eval_js('record(%s)' % (sec * 1000))
    # Decodifica o áudio em base64
    audio = b64decode(js_result.split(',')[1])
    # Salva o áudio em um arquivo
    file_name = 'entrada.wav'
    with open(file_name, 'wb') as f:
        f.write(audio)
    return f'/content/{file_name}'

print("Gravando...")
record_file = record()
display(Audio(record_file, autoplay=False))

# Reconhecimento de fala com Whisper (OpenAi)

In [ ]:
!pip install git+https://github.com/openai/whisper.git -q

In [ ]:
import whisper

model = whisper.load_model("small") # Carrega um modelo pré-treinado
result = model.transcribe(record_file, fp16=False, language=language) # Transcreve o arquivo de áudio indicado em 'record_file'
transcription = result["text"] # Extrai apenas o texto reconhecido do dicionário 'result'

print("Texto reconhecido:", transcription)

# Instalando o Ollama no Google Colab


In [ ]:
!sudo apt update # Atualiza a lista de pacotes disponíveis e suas versões no sistema
!sudo apt install -y pciutils # Instala o pacote 'pciutils', que contém ferramentas para identificar e gerenciar dispositivos PCI (como placas gráficas)
!apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh # Usa 'curl' para baixar um script de instalação hospedado em https://ollama.com/install.sh

In [ ]:
import threading # Permite criar e gerenciar threads (execuções paralelas)
import subprocess # Usado para executar comandos externos no sistema
import time # Fornece funções relacionadas a tempo, como pausas (sleep)

# Inicia o servidor do Ollama em segundo plano
def run_ollama_serve():
  subprocess.Popen(["ollama", "serve"])

# Permite que o servidor Ollama rode em paralelo sem travar o programa principal.
thread = threading.Thread(target=run_ollama_serve)
thread.start()
time.sleep(5)


# Integração com a API

In [ ]:
import subprocess # Importa o módulo subprocess, que permite executar comandos externos do sistema operacional

def asking_ollama(ask):
    process = subprocess.run(
        ["ollama", "run", "llama3.2"], # Define a versão do Ollama
        input=ask.encode("utf-8"), # Envia a pergunta (ask) como entrada, convertida para bytes UTF-8
        capture_output=True # Captura a saída do comando (stdout e stderr)
    )
    return process.stdout.decode("utf-8") # Retorna a saída padrão decodificada para string

response = asking_ollama(transcription) # Chama a função passando como entrada a transcrição obtida anteriormente (texto reconhecido do áudio)
print("Resposta do Ollama:", response)

#

# 4. Sintetizando a resposta como voz (gTTS)

In [ ]:
!pip install gTTS

In [ ]:
from gtts import  gTTS

gtts_object = gTTS(text=response, lang=language, slow=False) # Cria um objeto gTTS com a resposta gerada pelo Ollama e a língua que será sintetizada em voz (variável "language")

response_audio = "/content/record_file.wav" # Salva o áudio da resposta no arquivo especificado (pasta padrão do Google Colab)
gtts_object.save(response_audio)

display(Audio(response_audio, autoplay=True)) # Reproduz o áudio da resposta salvo no arquivo